In [1]:
!pip install -q pandas numpy requests lizard || pip install -q pandas numpy requests lizard --break-system-packages

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.5 MB/s eta 0:00:00


In [3]:
import subprocess
import os
import re
import time
import requests
import pandas as pd
import numpy as np
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed

HEADERS = {"User-Agent": "qm640-capstone"}
np.random.seed(42)
CHECKPOINT_FILE = "hadoop_mining_checkpoint.csv"
LARGE_FILE_CHAR_THRESHOLD = 250_000

# ----------------------------------------------------------------------
# 1. PARALLEL JIRA EXTRACTION
# ----------------------------------------------------------------------
def _fetch_jira_page(start_at: int, project_key: str, fields: str, jql: str) -> list:
    url = "https://issues.apache.org/jira/rest/api/2/search"
    params = {"jql": jql, "startAt": start_at, "maxResults": 100, "fields": fields}
    try:
        resp = requests.get(url, params=params, headers=HEADERS, timeout=30)
        resp.raise_for_status()
        return resp.json().get("issues", [])
    except Exception:
        return []

def extract_jira_issues_parallel(project_key: str, max_workers: int = 10) -> pd.DataFrame:
    url = "https://issues.apache.org/jira/rest/api/2/search"
    jql = f'project={project_key} AND resolution=Fixed ORDER BY resolutiondate ASC'
    fields = "created,resolutiondate,issuetype"

    resp = requests.get(url, params={"jql": jql, "startAt": 0, "maxResults": 1, "fields": fields}, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    total = resp.json().get("total", 0)
    print(f"Total JIRA issues to fetch for {project_key}: {total}")

    start_positions = range(0, total, 100)
    all_issues = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(_fetch_jira_page, pos, project_key, fields, jql): pos for pos in start_positions}
        for future in as_completed(futures):
            all_issues.extend(future.result())

    rows = []
    for issue in all_issues:
        f = issue.get("fields", {})
        rows.append({
            "issue_id": issue["key"],
            "project_name": project_key,
            "created": f.get("created"),
            "resolution_date": f.get("resolutiondate"),
            "issue_type": (f.get("issuetype") or {}).get("name"),
        })
    return pd.DataFrame(rows)

# ----------------------------------------------------------------------
# 2. GIT INDEXING & HELPER FUNCTIONS
# ----------------------------------------------------------------------
def ensure_repo(name, url, path):
    if not os.path.exists(path):
        print(f"Cloning real {name} repository...")
        subprocess.run(["git", "clone", "--filter=blob:none", url, path], check=True)
    return path

def build_issue_to_commit_index(repo_dir, prefix):
    result = subprocess.run(["git", "-C", repo_dir, "log", "--all", "--format=%H|%s"],
                              capture_output=True, text=True, timeout=300)
    index = defaultdict(list)
    pattern = re.compile(rf"{prefix}-\d+", re.IGNORECASE) # Fixed regex from \\d+ to \d+
    for line in result.stdout.splitlines():
        if "|" not in line:
            continue
        sha, msg = line.split("|", 1)
        for iid in pattern.findall(msg):
            index[iid.upper()].append(sha)
    return index

def build_commit_to_files_index(repo_dir, target_shas):
    files_by_commit = defaultdict(list)
    batch_size = 2000
    for i in range(0, len(target_shas), batch_size):
        batch = target_shas[i:i + batch_size]
        if not batch:
            continue
        result = subprocess.run(
            ["git", "-C", repo_dir, "log", "--name-only", "--format=COMMIT:%H", "--no-walk"]
            + batch + ["--", "*.java"],
            capture_output=True, text=True, timeout=120
        )
        current = None
        for line in result.stdout.splitlines():
            if line.startswith("COMMIT:"):
                current = line.replace("COMMIT:", "").strip()
            elif line.strip() and current:
                files_by_commit[current].append(line.strip())
    return files_by_commit

class BatchFileReader:
    def __init__(self, repo_dir):
        self.proc = subprocess.Popen(["git", "-C", repo_dir, "cat-file", "--batch"],
                                       stdin=subprocess.PIPE, stdout=subprocess.PIPE)

    def read(self, sha, path):
        self.proc.stdin.write(f"{sha}:{path}\n".encode("utf-8"))
        self.proc.stdin.flush()
        header = self.proc.stdout.readline().decode("utf-8", errors="replace")
        parts = header.split()
        if len(parts) < 2 or parts[1] == "missing":
            return ""
        try:
            size = int(parts[2])
        except (ValueError, IndexError):
            return ""
        content_bytes = self.proc.stdout.read(size)
        self.proc.stdout.read(1)
        if parts[1] != "blob":
            return ""
        return content_bytes.decode("utf-8", errors="replace")

    def close(self):
        try:
            self.proc.stdin.close()
            self.proc.wait()
        except Exception:
            pass

# ----------------------------------------------------------------------
# 3. WORKER FUNCTION FOR MULTI-CORE MINING
# ----------------------------------------------------------------------
def _process_commit_worker(task_tuple):
    row_dict, files, repo_dir = task_tuple
    import lizard

    reader = BatchFileReader(repo_dir)
    total_nloc, total_cc, total_fn, files_ok = 0, 0, 0, 0

    try:
        for path in files:
            content = reader.read(row_dict["sha"], path)
            if not content.strip() or len(content) > LARGE_FILE_CHAR_THRESHOLD:
                continue
            try:
                a = lizard.analyze_file.analyze_source_code(path, content)
            except Exception:
                continue
            total_nloc += a.nloc
            total_fn += len(a.function_list)
            for fn in a.function_list:
                total_cc += fn.cyclomatic_complexity
            files_ok += 1
    finally:
        reader.close()

    if files_ok == 0 or total_fn == 0:
        return None

    res = {
        "loc": total_nloc,
        "cyclomatic_complexity": total_cc / total_fn,
        "num_functions": total_fn,
        "num_files_changed": files_ok,
        "issue_id": row_dict["issue_id"],
        "project_name": "HADOOP",
        "era": row_dict["era"],
        "real_issue_type": row_dict["issue_type"],
        "defect_prone_strict": 1 if row_dict["issue_type"] == "Bug" else 0
    }
    return res

# ----------------------------------------------------------------------
# 4. MAIN PIPELINE EXECUTION
# ----------------------------------------------------------------------
def run_hadoop_pipeline():
    t0 = time.time()
    print("=== Extracting real Hadoop JIRA issues (Parallel) ===")
    jira_df = extract_jira_issues_parallel("HADOOP")
    jira_df["resolution_date_parsed"] = pd.to_datetime(jira_df["resolution_date"], errors="coerce", utc=True)
    jira_df["era"] = (jira_df["resolution_date_parsed"] >= "2023-01-01").map({True: "ai_era", False: "pre_ai"})
    print(f"[TIMING] JIRA extraction: {time.time()-t0:.1f}s, {len(jira_df)} real issues")

    hadoop_repo = ensure_repo("Hadoop", "https://github.com/apache/hadoop.git", "repos/hadoop")

    print("\n=== Matching real issues to commits ===")
    t1 = time.time()
    issue_index = build_issue_to_commit_index(hadoop_repo, "HADOOP")
    matched_rows = []
    for _, row in jira_df.iterrows():
        shas = issue_index.get(row["issue_id"].upper(), [])
        if shas:
            matched_rows.append({"issue_id": row["issue_id"], "sha": shas[0],
                                   "era": row["era"], "issue_type": row["issue_type"]})
    matched_df = pd.DataFrame(matched_rows)
    print(f"Real matches: {len(matched_df)} of {len(jira_df)} [TIMING] {time.time()-t1:.1f}s")

    print("\n=== Building real file-change index ===")
    t2 = time.time()
    target_shas = matched_df["sha"].tolist()
    files_index = build_commit_to_files_index(hadoop_repo, target_shas)
    print(f"[TIMING] File-index: {time.time()-t2:.1f}s")

    print("\n=== Mining real code metrics (Multi-Core Parallel) ===")
    already_done = set()
    rows = []
    if os.path.exists(CHECKPOINT_FILE):
        prev = pd.read_csv(CHECKPOINT_FILE)
        rows = prev.to_dict("records")
        already_done = set(prev["issue_id"])
        print(f"Resuming: {len(already_done)} already done")

    to_process = matched_df[~matched_df["issue_id"].isin(already_done)]

    tasks = []
    for _, row in to_process.iterrows():
        files = files_index.get(row["sha"], [])
        files = [f for f in files if f.endswith(".java") and "/test/" not in f]
        if files:
            tasks.append((row.to_dict(), files, hadoop_repo))

    num_workers = min(os.cpu_count() or 4, 8)
    print(f"Executing mining across {num_workers} CPU workers...")

    with ProcessPoolExecutor(max_workers=num_workers) as executor:
        futures = {executor.submit(_process_commit_worker, task): idx for idx, task in enumerate(tasks)}
        for count, future in enumerate(as_completed(futures)):
            try:
                res = future.result(timeout=15)
                if res:
                    rows.append(res)
            except Exception:
                continue

            if (count + 1) % 500 == 0:
                pd.DataFrame(rows).to_csv(CHECKPOINT_FILE, index=False)
                print(f"  [{count+1}/{len(tasks)}] checkpoint saved, {len(rows)} total usable")

    out = pd.DataFrame(rows)
    out.to_csv("hadoop_real_mined_dataset.csv", index=False)
    print(f"\nHADOOP COMPLETE: {len(out)} real mined samples")
    print(f"[TIMING] TOTAL: {time.time()-t0:.1f}s")
    return out

if __name__ == "__main__":
    run_hadoop_pipeline()


=== Extracting real Hadoop JIRA issues (Parallel) ===
Total JIRA issues to fetch for HADOOP: 10821
[TIMING] JIRA extraction: 2.6s, 10821 real issues

=== Matching real issues to commits ===
Real matches: 9961 of 10821 [TIMING] 2.3s

=== Building real file-change index ===
[TIMING] File-index: 94.5s

=== Mining real code metrics (Multi-Core Parallel) ===
Executing mining across 2 CPU workers...
  [500/6132] checkpoint saved, 499 total usable
  [1000/6132] checkpoint saved, 992 total usable
  [1500/6132] checkpoint saved, 1479 total usable
  [2000/6132] checkpoint saved, 1971 total usable
  [2500/6132] checkpoint saved, 2460 total usable
  [3000/6132] checkpoint saved, 2954 total usable
  [3500/6132] checkpoint saved, 3448 total usable
  [4000/6132] checkpoint saved, 3937 total usable
  [4500/6132] checkpoint saved, 4431 total usable
  [5000/6132] checkpoint saved, 4927 total usable
  [5500/6132] checkpoint saved, 5415 total usable
  [6000/6132] checkpoint saved, 5906 total usable

HADOO